# Set paths

In [ ]:
sample = "disease"
ref_model_path = "models/model_hlca_scanvi"
surgery_model_dir = f"surgery_models/hlca_{sample}_ext"
query_path = f'queries/hlca_queries/new/hlca_{sample}_ext.h5ad'
reference_raw = "models/model_hlca_scanvi/raw.h5ad"
reference_embedding = "models/model_hlca_scanvi/adata.h5ad"
combined_path = f"out/hlca_{sample}_combined.h5ad"

In [ ]:
import os

models = os.listdir(surgery_model_dir)
models.sort(key=lambda x: int(x.split("_")[-1]))

print(models)

# Create combined embedding

In [ ]:

import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=DeprecationWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
import scarches as sca
import scanpy as sc

In [ ]:
adata = sc.read_h5ad(query_path)

In [ ]:
adata_query = sca.models.SCANVI.prepare_query_anndata(
    adata = adata, reference_model = ref_model_path, inplace=False
)

sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="cell_ranger", batch_key="dataset")
adata_query.obs["scanvi_label"] = "unlabeled"


In [ ]:
reconstruction_loss = {}

print("get latent rep")

for model in models:
    surgery_model = sca.models.SCANVI.load(
        os.path.join(surgery_model_dir, model),
        adata_query
    )
    adata.obsm[model] = surgery_model.get_latent_representation(adata_query)
    reconstruction_loss[model]= surgery_model.get_reconstruction_error()["reconstruction_loss"]

adata.obs["ref_or_query"] = "query"

In [ ]:
reconstruction_loss

In [ ]:
print("load ref")

reference_raw = sc.read_h5ad(reference_raw)
reference = sc.read_h5ad(reference_embedding)

In [ ]:
reference.obsm_keys()

In [ ]:

for model in models:
    reference_raw.obsm[model] = reference.obsm["X_latent_qzm"] # you may want to change the embedding key here
reference_raw.obs["ref_or_query"] = "ref"

del reference

In [ ]:
combined = sc.concat((reference_raw, adata), index_unique=None,  join="outer")
combined.obs["ann_level_3"].fillna("Unknown", inplace=True)
combined.obs["ann_level_3"] = combined.obs["ann_level_3"].astype(str)
combined.write_h5ad(combined_path)

# Benchmark using scib

In [ ]:
combined = sc.read_h5ad(combined_path)


In [ ]:
print("pca")
sc.tl.pca(combined, use_highly_variable=False)

In [ ]:
combined.write_h5ad(combined_path)

In [ ]:
from scib_metrics.benchmark import *

biocons = BioConservation(False, False, True, True, False)
batchcor = BatchCorrection(False, False, True, True, True )
benchmark = Benchmarker(
    combined,
    batch_key='ref_or_query',
    label_key="ann_level_3",
    embedding_obsm_keys= models,
    n_jobs = 8,
    bio_conservation_metrics= biocons,
    batch_correction_metrics= batchcor,
    pre_integrated_embedding_obsm_key="X_pca"
)


In [ ]:
benchmark.prepare()

In [ ]:
benchmark.benchmark()

In [ ]:
print(benchmark.get_results(min_max_scale=False))

In [ ]:
benchmark.plot_results_table(min_max_scale=False)

# UMAP

In [ ]:
combined = sc.read_h5ad(combined_path)

In [ ]:
combined.obsm_keys()

In [ ]:
sc.pp.neighbors(combined, use_rep="epoch_250") # change to whichevery embedding you want to investigate

In [ ]:
sc.tl.umap(combined)

In [ ]:
sc.pl.umap(combined, color="ref_or_query", frameon=False, wspace=0.6)